# 18 提示工程（Prompt Engineering） / Prompt Engineering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BaiYuBY/use_token_to_learn/blob/main/docs/18-prompt-engineering/18-prompt-engineering.ipynb)

[📖 讲解 / Notes](https://baiyuby.github.io/use_token_to_learn/18-prompt-engineering/README.html) · [🎮 交互演示 / Playground](https://baiyuby.github.io/use_token_to_learn/18-prompt-engineering/interactive/prompting.html) · [🧑‍💻 GitHub](https://github.com/BaiYuBY/use_token_to_learn/tree/main/docs/18-prompt-engineering)

这个 notebook 由 `scripts/make_notebooks.py` 根据本章 `code/` 目录下的脚本自动生成，代码与脚本完全一致：每个函数或类是一个单元格，
可以逐个修改、重新运行。最后一个单元格运行本章的演示（带 assert 检查）。

Generated by `scripts/make_notebooks.py` from this chapter's `code/` scripts, with identical code: one cell per function or
class, so you can edit and re-run them one at a time. The last cell runs the chapter's demo (with its assert checks).

In [ ]:
# 环境 / setup: find the repository, or clone it (e.g. in Colab); some chapters import code from earlier ones
# 找到仓库，找不到就克隆一份（例如在 Colab 中）；有些章节会导入前面章节的代码
import subprocess
import sys
from pathlib import Path

CHAPTER = "18-prompt-engineering"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs" / CHAPTER).is_dir()), None)
if ROOT is None:
    ROOT = Path.cwd() / "use_token_to_learn"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/BaiYuBY/use_token_to_learn.git", str(ROOT)], check=True)
CODE = ROOT / "docs" / CHAPTER / "code"
sys.path.insert(0, str(CODE))
print("code:", CODE)

## `prompting.py`

```text
Prompt engineering in PyTorch / 用 PyTorch 观察提示工程.

Run / 运行:
    python docs/18-prompt-engineering/code/prompting.py

Covers / 包含:
    1. a tiny chat world: a system turn and a user turn, 28 everyday words, three tasks
       (what kind of thing is it, is it big, is it dangerous)
       一个小小的对话世界：系统轮和用户轮，28 个常见词，三个任务（是什么类别、大不大、危不危险）
    2. instructions vs examples: a task named in the prompt, or shown by a few examples whose label
       words are made up for each prompt (in-context learning)
       指令与示例：在提示里直接说任务，或者给几个示例，示例的标签词每次都是临时编的（上下文学习）
    3. what examples can and cannot do: coverage, and what the model guesses when no example matches
       示例能做什么、不能做什么：覆盖面，以及没有相符示例时模型怎么猜
    4. prompt injection: an instruction hidden in the user's text overrides the developer's system
       prompt; training an instruction hierarchy fixes that form of attack
       提示注入：藏在用户文本里的指令盖过开发者的系统提示；训练指令层级能挡住这种形式的攻击
    5. many-shot injection: examples instead of an instruction, a form the defence never saw
       多示例注入：用示例代替指令，这是防线没见过的形式
```

In [ ]:
__file__ = str(CODE / "prompting.py")  # the code below finds its data files from here / 代码据此定位数据文件

In [ ]:
import copy
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

### 1. A tiny chat world / 小小的对话世界

In [ ]:
KIND = {}
for _w in "cat dog mouse bee lion whale shark chicken fish cow".split():
    KIND[_w] = "animal"
for _w in "apple bread cake mushroom rice egg carrot cheese".split():
    KIND[_w] = "food"
for _w in "car truck house tree ship rock book phone knife fire".split():
    KIND[_w] = "thing"
BIG = set("lion whale shark cow car truck house tree ship".split())
DANGER = set("bee lion shark mushroom car truck knife fire".split())
OBJECTS = list(KIND)
KINDS = ["animal", "food", "thing"]
TASKS = ["kind?", "big?", "dangerous?"]
SYMBOLS = ["red", "blue", "green", "gold", "pink", "gray"]  # made-up label words for in-context learning / 临时标签词
ROLES = ["<sys>", "<user>"]  # role tokens: only the chat template can produce them / 角色标记：只有对话模板能生成
WORDS = ["<pad>"] + ROLES + [";", "sys", "yes", "no"] + KINDS + TASKS + SYMBOLS + OBJECTS
STOI = {w: i for i, w in enumerate(WORDS)}
ANSWERS = ["yes", "no"] + KINDS + SYMBOLS
ANS_IDS = torch.tensor([STOI[w] for w in ANSWERS])
OBJ_IDS = torch.tensor([STOI[w] for w in OBJECTS])
CTX = 48

In [ ]:
def answer(task, word, labels=None):
    """The right answer; labels renames the three kinds for in-context learning.
    正确答案；labels 用于上下文学习，把三个类别换成临时的标签词。"""
    if task == "kind?":
        return labels[KIND[word]] if labels else KIND[word]
    return "yes" if word in (BIG if task == "big?" else DANGER) else "no"

In [ ]:
def prompt(system=None, user=None, examples=(), query="cat", task="kind?", labels=None, inject_at="start", forge=None):
    """<sys> [system task] <user> [user task] word label ; word label ; ... query -> answer

    examples are labelled by `task` (the task the developer wants); `user` is an instruction typed in
    the user turn, at the start or just before the query. forge="escaped" puts the word "sys" in front
    of it: what a chat template turns a typed "<sys>" into, since user text can never produce a role
    token. Returns the words, ending with the right answer.
    示例按 task（开发者想要的任务）标注；user 是用户轮里的一条指令，放在开头或问题前。forge="escaped" 在它前面放单词
    "sys"：对话模板会把用户打出的 "<sys>" 变成这个普通词，因为用户文本永远生成不了角色标记。返回的词序列以正确答案结尾。"""
    seq = ["<sys>"] + ([system] if system else []) + ["<user>"]
    typed = (["sys"] if forge == "escaped" else []) + ([user] if user else [])
    if inject_at == "start":
        seq += typed
    for w in examples:
        seq += [w, answer(task, w, labels), ";"]
    if inject_at == "end":
        seq += typed
    return seq + [query, answer(task, query, labels)]

In [ ]:
def made_up_labels(rng):
    return dict(zip(KINDS, rng.sample(SYMBOLS, 3)))

In [ ]:
def training_prompt(rng, hierarchy=False):
    """The training mix. Half the prompts teach in-context learning: examples with made-up labels and
    no instruction. The rest name the task in the system or user turn. When the two turns disagree,
    the base model was taught that the latest instruction wins (like a chat model eager to help the
    user); the hierarchy fine-tune teaches that the system turn wins, also when the user writes "sys".
    训练数据：一半用于上下文学习（示例用临时标签，不给指令）；其余在系统轮或用户轮说明任务。两轮冲突时，基础模型学的是
    “最新的指令说了算”（像一个急于满足用户的对话模型）；指令层级微调学的是“系统轮说了算”，用户写了 "sys" 也一样。"""
    k = rng.choice([0, 1, 2, 3, 4, 6, 8])
    words = rng.sample(OBJECTS, k + 1)
    query = words.pop()
    r = rng.random()
    if r < 0.5:
        return prompt(examples=words or rng.sample(OBJECTS, 1), query=query, labels=made_up_labels(rng))
    task, other = rng.sample(TASKS, 2)
    if r < 0.75:
        return prompt(system=task, examples=words, query=query, task=task)
    if r < 0.88:
        return prompt(user=task, examples=words, query=query, task=task)
    winner = task if hierarchy else other
    forge = rng.choice([None, None, "escaped"]) if hierarchy else None
    return prompt(system=task, user=other, examples=words[:rng.choice([0, 1, 2])], query=query, task=winner, forge=forge)

In [ ]:
def to_ids(seqs):
    ids = torch.zeros(len(seqs), max(map(len, seqs)), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor([STOI[w] for w in s])
    return ids

### A small GPT (same layout as chapter 12's, so the page can run it) / 小型 GPT（与 12 章结构相同，交互页可以直接运行）

In [ ]:
class GPT(nn.Module):
    def __init__(self, d=64, layers=2, heads=4):
        super().__init__()
        self.embed, self.pos = nn.Embedding(len(WORDS), d), nn.Embedding(CTX, d)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout=0.0, batch_first=True, norm_first=True, activation="gelu")
        self.blocks = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)

    def forward(self, ids):
        T = ids.size(1)
        x = self.embed(ids) + self.pos(torch.arange(T))
        x = self.blocks(x, mask=nn.Transformer.generate_square_subsequent_mask(T), is_causal=True)
        return self.norm(x) @ self.embed.weight.T  # output layer tied to the embeddings / 输出层与嵌入共享权重

In [ ]:
def train(model, steps, hierarchy=False, lr=6e-3, seed=0):
    """Loss only on answers: every label that follows a word, i.e. each example's label and the final
    answer. 只在答案上算损失：每个单词后面的标签，即每个示例的标签和最后的答案。"""
    torch.manual_seed(seed)
    rng = random.Random(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    model.train()
    for step in range(steps):
        for g in opt.param_groups:
            g["lr"] = lr * min(1.0, (step + 1) / 50) * (0.1 + 0.9 * (1 - step / steps))
        ids = to_ids([training_prompt(rng, hierarchy) for _ in range(64)])
        logits, target = model(ids[:, :-1]), ids[:, 1:]
        mask = torch.isin(target, ANS_IDS) & torch.isin(ids[:, :-1], OBJ_IDS)
        loss = F.cross_entropy(logits[mask], target[mask])
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    return model

In [ ]:
@torch.no_grad()
def predict(model, seqs):
    """The model's answer (restricted to answer words) for prompts that end with the right answer.
    模型的回答（只在答案词里选）；输入的提示以正确答案结尾，预测时去掉它。"""
    ids = to_ids([s[:-1] for s in seqs])
    last = torch.tensor([len(s) - 2 for s in seqs])
    logits = model(ids)[torch.arange(len(seqs)), last][:, ANS_IDS]
    return [ANSWERS[i] for i in logits.argmax(-1).tolist()]

In [ ]:
def accuracy(model, seqs):
    return sum(p == s[-1] for p, s in zip(predict(model, seqs), seqs)) / len(seqs)

### 2-3. Instructions vs examples / 指令与示例

In [ ]:
def icl_prompts(rng, k, n, mode="random"):
    """In-context learning prompts. random: k random examples; one_each: one example of each kind
    (k = 3); missing: three examples of one kind and one of another, query of the third kind.
    上下文学习的提示。random：k 个随机示例；one_each：每类一个（k = 3）；missing：一类三个、另一类一个，问第三类。"""
    out = []
    for _ in range(n):
        labels = made_up_labels(rng)
        if mode == "random":
            words = rng.sample(OBJECTS, k + 1)
            query = words.pop()
        elif mode == "one_each":
            words = [rng.choice([w for w in OBJECTS if KIND[w] == c]) for c in KINDS]
            rng.shuffle(words)
            query = rng.choice([w for w in OBJECTS if w not in words])
        else:
            a, b, c = rng.sample(KINDS, 3)
            words = [rng.choice([w for w in OBJECTS if KIND[w] == a]) for _ in range(3)] + [rng.choice([w for w in OBJECTS if KIND[w] == b])]
            query = rng.choice([w for w in OBJECTS if KIND[w] == c])
        out.append((prompt(examples=words, query=query, labels=labels), words, labels))
    return out

In [ ]:
def covered(words, query):
    return KIND[query] in {KIND[w] for w in words}

### 4. Prompt injection / 提示注入

In [ ]:
ATTACKS = {"start": dict(inject_at="start"),                     # the form used in the fine-tune / 微调时用过的形式
           "end": dict(inject_at="end"),                         # a position the fine-tune never used / 微调没用过的位置
           "escaped": dict(inject_at="start", forge="escaped")}  # "sys" typed by the user / 用户打出来的 "sys"

In [ ]:
def injection(model, attack, n=300, seed=0):
    """The system turn names the task; the user turn contains a different instruction. Returns the share
    of answers that follow the system and the share that follow the injected instruction (counting only
    queries where the two answers differ). 系统轮指定任务，用户轮里藏着另一条指令。返回听系统的比例和听注入指令的比例。"""
    rng = random.Random(seed)
    sys_n = inj_n = total = 0
    while total < n:
        task, other = rng.sample(TASKS, 2)
        query = rng.choice(OBJECTS)
        if answer(task, query) == answer(other, query):
            continue
        p = predict(model, [prompt(system=task, user=other, query=query, task=task, **ATTACKS[attack])])[0]
        sys_n += p == answer(task, query)
        inj_n += p == answer(other, query)
        total += 1
    return sys_n / n, inj_n / n

In [ ]:
def example_injection(model, k, n=300, seed=1):
    """Many-shot injection: the system names a task, but the user turn holds k examples answered as a
    different task. Returns the shares following the system and following the examples.
    多示例注入：系统指定一个任务，用户轮里却有 k 个按另一个任务回答的示例。返回听系统的比例和跟着示例走的比例。"""
    rng = random.Random(seed)
    sys_n = inj_n = total = 0
    while total < n:
        task, other = rng.sample(TASKS, 2)
        words = rng.sample(OBJECTS, k + 1)
        query = words.pop()
        if answer(task, query) == answer(other, query):
            continue
        seq = ["<sys>", task, "<user>"]
        for w in words:
            seq += [w, answer(other, w), ";"]
        p = predict(model, [seq + [query, answer(task, query)]])[0]
        sys_n += p == answer(task, query)
        inj_n += p == answer(other, query)
        total += 1
    return sys_n / n, inj_n / n

In [ ]:
def user_instruction_accuracy(model, n=300, seed=0):
    """No system task: the user names it. Does the model still listen? 没有系统任务，由用户指定：模型还听吗？"""
    rng = random.Random(seed)
    seqs = []
    for _ in range(n):
        task = rng.choice(TASKS)
        seqs.append(prompt(user=task, query=rng.choice(OBJECTS), task=task))
    return accuracy(model, seqs)

In [ ]:
def train_models(steps=4000, finetune=400, seed=0):
    torch.manual_seed(seed)
    base = train(GPT(), steps, seed=seed)
    tuned = train(copy.deepcopy(base), finetune, hierarchy=True, lr=1e-3, seed=seed + 1)
    return base, tuned

In [ ]:
def main():
    print("== 1. a tiny chat world / 小小的对话世界 ==")
    rng = random.Random(0)
    print(" ".join(prompt(system="kind?", examples=["dog", "rock"], query="apple")))
    print(" ".join(prompt(examples=["dog", "rock", "egg"], query="lion", labels=made_up_labels(rng))))
    base, tuned = train_models()

    print("\n== 2. instructions vs examples / 指令与示例 ==")
    for task in TASKS:
        seqs = [prompt(system=task, query=w, task=task) for w in OBJECTS]
        print(f"  instruction only, {task:10s}: {accuracy(base, seqs):.0%}")
        assert accuracy(base, seqs) > 0.95
    curve = {}
    for k in [0, 1, 2, 3, 4, 6, 8]:
        items = icl_prompts(random.Random(10 + k), k, 400)
        curve[k] = accuracy(base, [s for s, _, _ in items])
        hit = [s for s, w, _ in items if covered(w, s[-2])]
        print(f"  {k} examples with made-up labels: {curve[k]:.0%}"
              + (f"  (query's kind among the examples: {accuracy(base, hit):.0%} of {len(hit)})" if hit and k else ""))
    one_each = accuracy(base, [s for s, _, _ in icl_prompts(random.Random(30), 3, 400, "one_each")])
    print(f"  3 examples, one of each kind: {one_each:.0%}  (vs {curve[3]:.0%} for 3 random ones)")
    assert curve[8] > curve[1] + 0.3 and one_each > curve[3] + 0.1

    print("\n== 3. no matching example: what does it guess? / 没有相符的示例时，它猜什么 ==")
    items = icl_prompts(random.Random(40), 4, 400, "missing")
    preds = predict(base, [s for s, _, _ in items])
    majority = sum(p == labels[KIND[w[0]]] for p, (_, w, labels) in zip(preds, items)) / len(items)
    latest = sum(p == labels[KIND[w[-1]]] for p, (_, w, labels) in zip(preds, items)) / len(items)
    unused = sum(p in SYMBOLS and p not in labels.values() for p, (_, w, labels) in zip(preds, items)) / len(items)
    print(f"  3 examples of kind A, then 1 of kind B, query of kind C: answers A's label {majority:.0%}, "
          f"B's label {latest:.0%}, an unused label {unused:.0%}")

    print("\n== 4. prompt injection / 提示注入 ==")
    for name, model in [("base model", base), ("+ hierarchy fine-tune", tuned)]:
        cells = []
        for attack in ATTACKS:
            follow_sys, follow_inj = injection(model, attack)
            cells.append(f"{attack} {follow_inj:.0%}")
        print(f"  {name:22s} injected instruction obeyed: " + ", ".join(cells)
              + f";  user-only instruction followed {user_instruction_accuracy(model):.0%}")
    assert injection(base, "start")[1] > 0.8          # the latest instruction wins / 最新的指令说了算
    assert injection(tuned, "start")[1] < 0.1         # the hierarchy holds / 层级生效
    assert injection(tuned, "escaped")[1] < 0.1       # an escaped role word is just a word / 转义后的角色词只是普通单词
    assert user_instruction_accuracy(tuned) > 0.9     # users can still give instructions / 用户仍然可以下指令

    print("\n== 5. many-shot injection: examples instead of an instruction / 多示例注入：用示例代替指令 ==")
    shots = {}
    for k in [1, 2, 4, 8]:
        shots[k] = example_injection(tuned, k)
        print(f"  {k} examples answered as another task: follows system {shots[k][0]:.0%}, follows the examples {shots[k][1]:.0%}")

### 运行演示 / Run the demo (`prompting.py`)

In [ ]:
main()

## 动手试试 / Try it yourself

改一改上面任意一个函数或参数，重新运行它所在的单元格，再运行 `main()` 看结果有什么变化。
assert 失败说明你的改动改变了程序检查的性质，这正是值得思考的地方。

Change any function or parameter above, re-run its cell, then run `main()` again. If an assert fails, your change broke a
property the script checks, which is exactly what is worth thinking about.